In [25]:
# !conda install -c conda-forge python-docx -y
# !pip install docx2pdf

In [26]:
from dotenv import load_dotenv
load_dotenv()


True

In [27]:
from datetime import datetime
import re
import json
import bs4
from pathlib import Path

from docx import Document
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import WebBaseLoader

# ----------------------------
# 0) Setup
# ----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
today = datetime.now().strftime("%B %d, %Y")

# Load job URL from link.text (one folder up)
job_url_path = (Path.cwd().parent / "link.text")
JOB_URL = job_url_path.read_text(encoding="utf-8").strip()

# ----------------------------
# 1) Load job posting text
# ----------------------------
def load_job_text(url: str) -> str:
    # Try to grab a broad set of content (LinkedIn is dynamic; this may be partial)
    loader = WebBaseLoader(web_paths=(url,))
    docs = loader.load()
    raw = "\n".join(d.page_content for d in docs)
    # Basic cleanup
    raw = re.sub(r"\s+", " ", raw).strip()
    return raw

job_text = load_job_text(JOB_URL)

# ----------------------------
# 2) Extract company / location / title (LLM-based, more robust than regex)
def _parse_llm_json(text: str) -> dict:
    text = (text or "").strip()

    # Handle fenced blocks like ```json ... ```
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_-]*\n", "", text)
        text = re.sub(r"\n```$", "", text)

    # Try direct JSON parse first
    try:
        return json.loads(text)
    except Exception:
        pass

    # Fallback: parse first JSON object found in the response
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass

    return {}


def extract_job_fields(job_text: str) -> dict:
    prompt = f"""
Extract job details from the text below.

Return ONLY a valid JSON object with these keys:
- "job_title"
- "company"
- "location"

If a field is not present, return it as "".

TEXT:
{job_text}
"""
    resp = llm.invoke(prompt).content
    parsed = _parse_llm_json(resp)

    company = (parsed.get("company") or "").strip()
    job_title = (parsed.get("job_title") or "").strip()
    location = (parsed.get("location") or "").strip()

    # Heuristic fallback from raw text when the model output is malformed/empty
    if not company or not job_title or not location:
        title_match = re.search(r"([A-Z][A-Za-z0-9&/,+\-. ]{2,80})\s+at\s+([A-Z][A-Za-z0-9&/,+\-. ]{1,80})", job_text)
        if title_match:
            job_title = job_title or title_match.group(1).strip()
            company = company or title_match.group(2).strip()

        loc_match = re.search(r"(Remote|Hybrid|On[- ]site|[A-Z][a-zA-Z]+,\s*[A-Z]{2})", job_text)
        if loc_match:
            location = location or loc_match.group(1).strip()

    return {
        "job_title": job_title,
        "company": company,
        "location": location,
    }

fields = extract_job_fields(job_text)
job_title = (fields.get("job_title") or "").strip() or "UnknownJobTitle"
company = (fields.get("company") or "").strip() or "UnknownCompany"
location = (fields.get("location") or "").strip() or "UnknownLocation"

print(f"Extracted company: {company}")
print(f"Extracted job title: {job_title}")
print(f"Extracted location: {location}")

# ----------------------------
# 3) Rewrite cover letter using your existing structure (cl as the template)
# ----------------------------
def generate_cover_letter_from_template(template_cl: str, job_text: str, company: str, location: str, job_title: str) -> str:
    prompt = f"""
You are rewriting a cover letter.

GOAL:
- Keep the SAME overall structure and tone as the template letter.
- Update placeholders in square brackets (like [Date], [Company], [Location], etc.).
- Replace [Date] with: {today}
- Update company/location/job title using:
  - Company: {company}
  - Location: {location}
  - Job Title: {job_title}
- Rewrite ONLY the BODY paragraphs to match the job requirements from the job text. Body ends with "Thank you for your time and consideration."
- Keep it internship-appropriate (no exaggerated senior claims).
- Keep it to ~3 body paragraphs, clear and specific, with relevant skills + impact.

JOB TEXT (requirements):
{job_text}

TEMPLATE COVER LETTER:
{template_cl}

OUTPUT:
Return the final cover letter as plain text, preserving paragraph breaks.
"""
    return llm.invoke(prompt).content


def enforce_company_mentions(letter_text: str, template_cl: str, company: str) -> str:
    if not company:
        return letter_text

    out = letter_text

    # Replace bracketed placeholders like [Company], [company ex: ...]
    out = re.sub(r"\[\s*company[^\]]*\]", company, out, flags=re.IGNORECASE)

    # Replace template example company if present in template text
    m = re.search(r"company\s*ex:\s*([^\]\n]+)", template_cl, flags=re.IGNORECASE)
    if m:
        template_company = m.group(1).strip()
        if template_company:
            out = re.sub(re.escape(template_company), company, out, flags=re.IGNORECASE)

    # Common fallback from current template sample
    out = re.sub(r"Voloridge\s+Health", company, out, flags=re.IGNORECASE)

    return out

def load_docx_text(path: Path) -> str:
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text).strip()

def write_paragraph_files(paragraphs: list[str], base_dir: Path) -> list[Path]:
    paths = []
    for i, paragraph in enumerate(paragraphs, start=1):
        path = base_dir / f"parent_folder.par{i}.txt"
        path.write_text(paragraph, encoding="utf-8")
        paths.append(path)
    return paths

def read_paragraph_files(base_dir: Path) -> list[str]:
    paths = sorted(base_dir.glob("parent_folder.par*.txt"), key=lambda p: int(re.search(r"(\d+)", p.stem).group(1)))
    return [path.read_text(encoding="utf-8").strip() for path in paths if path.read_text(encoding="utf-8").strip()]

cl = load_docx_text(Path.cwd().parent / "Uranbileg_CLetter.docx")
generated_letter = generate_cover_letter_from_template(
    template_cl=cl,
    job_text=job_text,
    company=company,
    location=location,
    job_title=job_title,
)
generated_letter = enforce_company_mentions(generated_letter, cl, company)

generated_paragraphs = [p.strip() for p in generated_letter.split("\n") if p.strip()]
paragraph_files = write_paragraph_files(generated_paragraphs, Path.cwd().parent)
print(f"Wrote {len(paragraph_files)} paragraph files to {Path.cwd().parent}")

# Re-load paragraphs from files so edits there are reflected in final output
final_paragraphs = read_paragraph_files(Path.cwd().parent)
final_cover_letter = "\n\n".join(final_paragraphs)

print(final_cover_letter)


Extracted company: Sundt Construction
Extracted job title: Support Group Intern - Data Engineering
Extracted location: Tempe, AZ
Wrote 14 paragraph files to /Users/uranbileg/Documents/JOB
URANBILEG ENKHJARGAL

Worcester, MA  (774) 351-8585  UEnkhjargal@clarku.edu  linkedin.com/in/uranbileg/

March 06, 2026

Sundt Construction

Tempe, AZ

Dear Hiring Team,

I am excited to apply for the Support Group Intern - Data Engineering position at Sundt Construction. I am currently pursuing a Master’s degree in Computer Science at Clark University, where I focus on data management, database development, and data analytics.

In my academic projects, I have developed and maintained databases and data pipelines, utilizing SQL and Python to create efficient ETL processes. I have collaborated with peers to troubleshoot and optimize existing solutions, ensuring high data quality and performance. This hands-on experience has equipped me with the skills to contribute effectively to the implementation of 

In [28]:
from docx import Document
from docx.shared import Pt
from pathlib import Path
import re

# Clean file name (keep letters/numbers/underscore)
def clean_filename(text):
    text = (text or "").strip()
    text = re.sub(r"[^A-Za-z0-9\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

# Extract non-empty paragraphs from generated letter
def extract_body_paragraphs(letter_text: str):
    lines = [ln.strip() for ln in letter_text.split("\n") if ln.strip()]
    if not lines:
        return []

    start_idx = 0
    end_idx = len(lines)

    for i, ln in enumerate(lines):
        if ln.lower().startswith("dear "):
            start_idx = i + 1
            break

    for i in range(start_idx, len(lines)):
        if lines[i].lower().startswith("sincerely"):
            end_idx = i
            break

    body = [ln for ln in lines[start_idx:end_idx] if ln]
    return body if body else lines

generated_paragraphs = extract_body_paragraphs(final_cover_letter)

template_path = Path.cwd().parent / "Uranbileg_CLetter.docx"
doc = Document(template_path)

# Keep header/contact/signature format from original template, replace only letter content
# These indexes match the current template structure
idx_date = 3
idx_company = 5
idx_location = 7
idx_salutation = 9
idx_body_start = 11
idx_closing = 18

if len(doc.paragraphs) <= idx_closing:
    raise ValueError("Template format changed. Please check paragraph indexes in Uranbileg_CLetter.docx")

doc.paragraphs[idx_date].text = today
doc.paragraphs[idx_company].text = company or ""
doc.paragraphs[idx_location].text = location or ""
doc.paragraphs[idx_salutation].text = "Dear Hiring Team,"

# Fill body paragraphs while preserving original paragraph spacing/layout
body_slots = [11, 13, 15]
for i, p_idx in enumerate(body_slots):
    doc.paragraphs[p_idx].text = generated_paragraphs[i] if i < len(generated_paragraphs) else ""

# If model returned more than 3 body paragraphs, append extras before closing
for extra in generated_paragraphs[len(body_slots):]:
    doc.paragraphs[idx_closing].insert_paragraph_before(extra)

# Make everything from the date line onward Calibri 11
for p in doc.paragraphs[idx_date:]:
    for run in p.runs:
        run.font.name = "Calibri"
        run.font.size = Pt(11)

company_clean = clean_filename(company)
job_clean = clean_filename(job_title)

filename = f"Uranbileg_CL_{company_clean}_{job_clean}.docx"
save_path = Path.cwd().parent / filename

doc.save(save_path)

pdf_path = save_path.with_suffix(".pdf")
try:
    from docx2pdf import convert
    convert(str(save_path), str(pdf_path))
    print(f"PDF saved to: {pdf_path}")
except Exception as exc:
    print("PDF export skipped. Install docx2pdf to enable PDF output.")
    print(f"Reason: {exc}")

print(f"Company: {company}")
print(f"Job Title: {job_title}")
print(f"Cover letter saved to: {save_path}")


/opt/anaconda3/envs/PY3_10_GENAI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1/1 [00:38<00:00, 38.90s/it]

PDF saved to: /Users/uranbileg/Documents/JOB/Uranbileg_CL_Sundt_Construction_Support_Group_Intern_-_Data_Engineering.pdf
Company: Sundt Construction
Job Title: Support Group Intern - Data Engineering
Cover letter saved to: /Users/uranbileg/Documents/JOB/Uranbileg_CL_Sundt_Construction_Support_Group_Intern_-_Data_Engineering.docx
